In [ ]:
# import required libs
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
import TCP_IP_UTILS
import Utils
import time

In [ ]:
#Add power supply code here -> DC Sources
import sys
import pyvisa
sys.path.append(r"../PowerSupply")
import PS_Utils
Device_ID_DC = "USB0::0x2A8D::0x1202::MY59003914::0::INSTR"
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA resources:")
for r in resources:
        print(" -", r)
device_dc = PS_Utils.connect_E36313A(Device_ID_DC)
DIG_LS = 1
ANA_LS = 2
MAIN = 3

In [ ]:
#connect to ethernet socket 
HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)
time.sleep(1) #Wait for IOs to settle

In [ ]:
#First Turn on Digital Leval Shifters
PS_Utils.set_voltage_E36313A(device_dc,DIG_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,DIG_LS))
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

In [ ]:
#Turn on Main Chip PS
PS_Utils.set_voltage_E36313A(device_dc,MAIN,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,MAIN))

In [ ]:
#Turn on CLK 
Device_ID_CLK = "USB0::0x0957::0x1F01::MY53270560::0::INSTR"
device_clk = PS_Utils.connect_N5173B(Device_ID_CLK)
PS_Utils.set_voltage_N5173B(device_clk,0.45)
PS_Utils.set_frequency_N5173B(device_clk,1000000000)
PS_Utils.enable_output_N5173B(device_clk)

In [ ]:
#Initialise SRAM with default data.
path = r"..\Utils\CalibrationData.xlsx"
write_data_default = Utils.get_Default_Write_Data(path).tolist()
ret_data = Utils.Write_SRAM(client_socket,write_data_default,1)
print(ret_data)

In [ ]:
#Turn on Analog PS
PS_Utils.set_voltage_E36313A(device_dc,ANA_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,ANA_LS))

In [39]:
#combined_value_array = [Utils.TDC_OUT_SC_LM["value"],Utils.TDC_OUT_SC_RM["value"],Utils.DL_EN_SC_LM["value"],Utils.DL_EN_SC_RM["value"],Utils.WL_SC["value"],Utils.SA_OUT_SC["value"],Utils.WRITE_DATA_SC["value"],Utils.CONTROL_LOGIC_SC["value"]]
combined_id_array = [Utils.TDC_OUT_SC_LM["id"],Utils.TDC_OUT_SC_RM["id"],Utils.DL_EN_SC_LM["id"],Utils.DL_EN_SC_RM["id"],Utils.WL_SC["id"],Utils.SA_OUT_SC["id"],Utils.WRITE_DATA_SC["id"],Utils.CONTROL_LOGIC_SC["id"]]
combined_len_array = [Utils.TDC_OUT_SC_LM["len"],Utils.TDC_OUT_SC_RM["len"],Utils.DL_EN_SC_LM["len"],Utils.DL_EN_SC_RM["len"],Utils.WL_SC["len"],Utils.SA_OUT_SC["len"],Utils.WRITE_DATA_SC["len"],Utils.CONTROL_LOGIC_SC["len"]]

In [41]:
#Set scan chain ID this will also be done in C. Here we have selected Control. So control EN needs to be off. Ensure Same for each scan chain to avoid toggles across chip. By Default Control EN is off. So no need to do again.
for sc in range(0,8):
    #Call Scan IN here with data
    scan_id = combined_id_array[sc]
    scan_len = combined_len_array[sc]
    #scan_data = np.ones(scan_len,dtype=int).tolist()
    #scan_data = np.zeros(scan_len,dtype=int).tolist()
    #scan_data = np.array([i % 2 for i in range(scan_len)], dtype=int).tolist()
    scan_data = np.array([(i+1) % 2 for i in range(scan_len)], dtype=int).tolist()
    ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
    print(ret_data)
    ret_data = Utils.Set_ChipIO_to_Default(client_socket)
    print(ret_data)

[128, 2]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]
Default Signals Set
[128]
[128, 2]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]
Default Signals Set
[128]
[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]
Default Signals Set
[128]
[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]
Default Signals Set
[128]
[132, 4]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]
Default Signals Set
[128]
[40, 1]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]
Default Signals Set
[128]
[40, 1]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received


In [42]:
scan_out = []
for sc in range(0,8):
    #Call Scan OUT here with data
    scan_id = combined_id_array[sc]
    scan_len = combined_len_array[sc]
    ret_data = Utils.ScanOUT_Data(client_socket,scan_id,scan_len,1)
    print(len(ret_data))
    scan_out.append(ret_data)
    #Set IO to default and exit    
    ret_data = Utils.Set_ChipIO_to_Default(client_socket)
    print(ret_data)

[128, 2]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 4
Received Data Size: 80
All Data Received
80
Default Signals Set
[128]
[128, 2]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 4
Received Data Size: 80
All Data Received
80
Default Signals Set
[128]
[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 4
Received Data Size: 12
All Data Received
12
Default Signals Set
[128]
[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 4
Received Data Size: 12
All Data Received
12
Default Signals Set
[128]
[132, 4]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 4
Received Data Size: 148
All Data Received
148
Default Signals Set
[128]
[40, 1]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 4
Received Data Size: 40
All Data Received
40
Default Signals Set
[128]
[40, 1]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 4
Received Data Size: 40
All Data Received
40
Defaul

In [46]:
arr = scan_out[4]
print(arr)

unique_elements = np.unique(arr)
print(unique_elements)


[85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 5, 0, 0, 0]
[ 0  5 85]


In [40]:
#Set IO to default and exit    
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

Default Signals Set
[128]


In [32]:
signal_array = [Utils.CLK_B,Utils.CLK_A]
value_array = [1,1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received


In [38]:
signal_array = [Utils.SCN_IN]
value_array = [0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received


In [47]:
PS_Utils.kill_channel_E36313A(device_dc,ANA_LS)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,MAIN)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,DIG_LS)
PS_Utils.disconnect_E36313A(device_dc)
time.sleep(0.3)
PS_Utils.kill_N5173B(device_clk)
PS_Utils.disconnect_N5173B(device_clk)

0